In [7]:
import pandas as pd
import numpy as np
df_model = pd.read_csv(
    "C:/Users/Mayur/CRYPTOCURRECNY VOLATILITY PROJECT/data/processed\model_df.csv"
)

df_model.head()

,volatility_7d,volatility_30d,daily_return,target_volatility,date,crypto_name
0,0.060476,18.606026,0.031970,0.059793,2020-11-01,Aave
1,0.059793,0.076802,-0.030197,0.060960,2020-11-02,Aave
2,0.060960,0.077142,-0.056799,0.075090,2020-11-03,Aave
3,0.075090,0.080150,0.103871,0.106530,2020-11-05,Aave
4,0.106530,0.085132,0.242341,0.115043,2020-11-06,Aave


In [8]:
df_model['date'] = pd.to_datetime(df_model['date'])
df_model = df_model.sort_values(
    ["date", "crypto_name"]
).reset_index(drop=True)

df_model.head()

,volatility_7d,volatility_30d,daily_return,target_volatility,date,crypto_name
0,0.025356,0.025711,-0.006562,0.021338,2013-06-04,Bitcoin
1,0.033567,0.041239,0.051795,0.033429,2013-06-04,Litecoin
2,0.021338,0.024945,0.001894,0.021783,2013-06-05,Bitcoin
3,0.033429,0.039937,-0.001500,0.033092,2013-06-05,Litecoin
4,0.021783,0.025605,-0.030004,0.025935,2013-06-06,Bitcoin


In [9]:
dates = sorted(df_model["date"].unique())

split_index = int(len(dates) * 0.8)
split_date = dates[split_index]

train_df = df_model[df_model["date"] < split_date]
test_df = df_model[df_model["date"] >= split_date]

In [11]:
features = [
    "volatility_7d",
    "volatility_30d",
    "daily_return"
]
target = "target_volatility"

X_train = train_df[features]
X_test = test_df[features]

y_train = train_df[target]
y_test = test_df[target]

In [12]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(
    tscv.split(X_train), start=1
):
    print(f"Fold {fold}")
    print("Training samples:", len(train_idx))
    print("Validation samples:", len(val_idx))
    print()

Fold 1
Training samples: 6535
Validation samples: 6531

Fold 2
Training samples: 13066
Validation samples: 6531

Fold 3
Training samples: 19597
Validation samples: 6531

Fold 4
Training samples: 26128
Validation samples: 6531

Fold 5
Training samples: 32659
Validation samples: 6531



In [13]:

# Get unique dates in chronological order
unique_dates = np.sort(df_model["date"].unique())

# Create 5 chronological folds
tscv = TimeSeriesSplit(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(
    tscv.split(unique_dates), start=1
):
    train_dates = unique_dates[train_idx]
    val_dates = unique_dates[val_idx]

    print(f"Fold {fold}")
    print("Training dates:", len(train_dates))
    print("Validation dates:", len(val_dates))
    print("Training ends:", train_dates[-1])
    print("Validation starts:", val_dates[0])
    print()

Fold 1
Training dates: 537
Validation dates: 536
Training ends: 2014-11-22T00:00:00.000000000
Validation starts: 2014-11-23T00:00:00.000000000

Fold 2
Training dates: 1073
Validation dates: 536
Training ends: 2016-05-11T00:00:00.000000000
Validation starts: 2016-05-12T00:00:00.000000000

Fold 3
Training dates: 1609
Validation dates: 536
Training ends: 2017-10-29T00:00:00.000000000
Validation starts: 2017-10-30T00:00:00.000000000

Fold 4
Training dates: 2145
Validation dates: 536
Training ends: 2019-04-18T00:00:00.000000000
Validation starts: 2019-04-19T00:00:00.000000000

Fold 5
Training dates: 2681
Validation dates: 536
Training ends: 2020-10-11T00:00:00.000000000
Validation starts: 2020-10-12T00:00:00.000000000



In [18]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Define the models
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42,
        objective="reg:squarederror",
        n_jobs=-1
    )
}

results = []

# Cross-validation
for fold, (train_idx, val_idx) in enumerate(
    tscv.split(unique_dates), start=1
):
    train_dates = unique_dates[train_idx]
    val_dates = unique_dates[val_idx]

    fold_train = df_model[df_model["date"].isin(train_dates)]
    fold_val = df_model[df_model["date"].isin(val_dates)]

    X_fold_train = fold_train[features]
    y_fold_train = fold_train["target_volatility"]

    X_fold_val = fold_val[features]
    y_fold_val = fold_val["target_volatility"]

    for name, model in models.items():
        model.fit(X_fold_train, y_fold_train)

        predictions = model.predict(X_fold_val)

        mae = mean_absolute_error(y_fold_val, predictions)
        rmse = np.sqrt(mean_squared_error(y_fold_val, predictions))

        results.append({
            "Model": name,
            "Fold": fold,
            "MAE": mae,
            "RMSE": rmse
        })

# Display results
results_df = pd.DataFrame(results)

print(results_df)

                Model  Fold       MAE      RMSE
0   Linear Regression     1  0.006565  0.012013
1       Random Forest     1  0.008845  0.014894
2             XGBoost     1  0.007925  0.013076
3   Linear Regression     2  0.008184  0.019630
4       Random Forest     2  0.009363  0.021489
5             XGBoost     2  0.009011  0.022658
6   Linear Regression     3  0.008577  0.017724
7       Random Forest     3  0.009692  0.018863
8             XGBoost     3  0.009223  0.020874
9   Linear Regression     4  0.006854  0.019708
10      Random Forest     4  0.007753  0.027962
11            XGBoost     4  0.007065  0.027992
12  Linear Regression     5  0.016135  0.552625
13      Random Forest     5  0.027619  1.048879
14            XGBoost     5  0.027281  1.058527


In [19]:
summary_df = (
    results_df
    .groupby("Model")[["MAE", "RMSE"]]
    .mean()
    .sort_values("MAE")
)

summary_df

,MAE,RMSE
Model,,
Linear Regression,0.009263,0.124340
XGBoost,0.012101,0.228625
Random Forest,0.012654,0.226418


In [20]:
# Get the final validation period
fold5_dates = unique_dates[
    list(tscv.split(unique_dates))[-1][1]
]

fold5_df = df_model[
    df_model["date"].isin(fold5_dates)
]

# Check target statistics
fold5_df["target_volatility"].describe()

count    27523.000000
mean         0.083929
std          1.070688
min          0.000000
25%          0.036532
50%          0.055797
75%          0.081941
max         67.042805
Name: target_volatility, dtype: float64

In [21]:
# Display the 10 highest target volatility values
fold5_df.nlargest(
    10, "target_volatility"
)[
    ["date", "crypto_name", "volatility_7d",
     "volatility_30d", "daily_return", "target_volatility"]
]

,date,crypto_name,volatility_7d,volatility_30d,daily_return,target_volatility
48830,2021-01-31,Shiba Inu,67.013455,32.371664,177.327290,67.042805
49075,2021-02-05,Shiba Inu,67.019515,32.374193,0.000000,67.036015
48977,2021-02-03,Shiba Inu,67.016194,32.373906,-0.500000,67.032700
49026,2021-02-04,Shiba Inu,67.032700,32.372566,0.000000,67.019515
48928,2021-02-02,Shiba Inu,66.997537,32.368980,1.000000,67.016194
48781,2021-01-29,Shiba Inu,0.252205,0.227747,0.263367,67.013455
48879,2021-02-01,Shiba Inu,67.042805,32.374770,-0.666667,66.997537
53404,2021-05-06,Shiba Inu,0.198543,0.520160,-0.105263,3.537394
53454,2021-05-08,Shiba Inu,3.537394,1.742617,9.313725,3.534004
53704,2021-05-14,Shiba Inu,3.521062,1.759850,0.042817,3.524984
